# Phase 7A: Temporal Robustness and Model Aging

This notebook is a presentation layer. It reads the generated Phase 7A artifacts; training and cascade logic remain under `src/`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'results' / 'experiments' / 'phase7a'
results = json.loads((OUT / 'phase7a_results.json').read_text(encoding='utf-8'))
periods = pd.DataFrame(results['period_definitions'])
metrics = pd.read_csv(OUT / 'phase7a_period_metrics.csv')
predictions = pd.read_csv(OUT / 'phase7a_predictions.csv')
routing = pd.read_csv(OUT / 'phase7a_routing_diagnostics.csv')

## Period definitions

In [ ]:
periods

## Forward-only metrics

In [ ]:
metric_columns = ['prediction_period', 'precision', 'recall', 'f1', 'pr_auc', 'roc_auc', 'balanced_accuracy', 'tn', 'fp', 'fn', 'tp']
metrics[metric_columns]

In [ ]:
metrics.set_index('prediction_period')[['precision', 'recall', 'f1', 'pr_auc', 'balanced_accuracy']].plot(
    marker='o', ylim=(0, 1), title='Frozen MulCas performance by prediction period'
)

## Routing diagnostics

In [ ]:
predictions.groupby(['period', 'accepted_viewer']).size().unstack(fill_value=0)

Metric changes are descriptive temporal variation. They do not by themselves prove concept drift, and no Phase 7A result is used to alter the frozen model.

# Phase 7B: Frozen MulCas component/view ablation

These tables present predeclared ablations only. They are not model-selection results.

In [ ]:
OUT_7B = ROOT / 'results' / 'experiments' / 'phase7b'
ablation_metrics = pd.read_csv(OUT_7B / 'phase7b_ablation_metrics.csv')
ablation_deltas = pd.read_csv(OUT_7B / 'phase7b_metric_deltas.csv')
ablation_routing = pd.read_csv(OUT_7B / 'phase7b_routing.csv')
frozen_viewers = json.loads((OUT_7B / 'phase7b_frozen_viewers.json').read_text(encoding='utf-8'))

## Actual frozen viewer sequence

In [ ]:
pd.DataFrame(frozen_viewers['viewers'])

## Ablation metrics and deltas

In [ ]:
ablation_metrics[['ablation_id', 'prediction_period', 'precision', 'recall', 'f1', 'pr_auc', 'balanced_accuracy', 'tn', 'fp', 'fn', 'tp']]

In [ ]:
ablation_deltas.pivot(index='ablation_id', columns='prediction_period', values='delta_f1')

In [ ]:
ablation_routing.pivot_table(index=['ablation_id', 'prediction_period'], columns='route', values='count', fill_value=0)